In [1]:
import json
import numpy as np
import pandas as pd
tgt_langs = [
    "Arabic",
    "Chinese",
    "French",
    "Japanese",
    "Russian",
]

models = {
    'nllb': 'nllb',
    'seamless': 'seamless',
    'gpt4omini': 'gpt-4o-mini',
    'aya_old': 'aya-23-8B',
    'aya': 'aya-expanse-8B'
}

eval_data = [json.loads(i) for i in open("/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/eval_data_google.jsonl", 'r').readlines()]

In [2]:
methods = {
    '_hard_replace_morphology': 'Morphologically Correct Word Alignment',
    '_prompt_gpt4omini': 'Prompting Refinement'
}

examples = {}

for model, model_name in models.items():
    
    # load data from a method
    method_data_hard = [json.loads(i) for i in open(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/output/{model}_hard_replace_morphology.jsonl", 'r').readlines()]
    
    method_data_prompt = [json.loads(i) for i in open(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/output/{model}_prompt_gpt4omini.jsonl", 'r').readlines()]
    
    method_data_hard_res = json.load(open(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/eval/{model}_hard_replace_morphology_comet.json", 'r'))
    
    method_data_prompt_res = json.load(open(f"/home/jiaruil5/multilingual/multilingual-model-card/multilingualmc/dataset/eval/{model}_prompt_gpt4omini_comet.json", 'r'))
    
    examples_method = {}
    
    for lang in tgt_langs:
        method_data_hard_res[lang] = np.array(method_data_hard_res[f'{lang}_scores'])
        method_data_prompt_res[lang] = np.array(method_data_prompt_res[f'{lang}_scores'])
        
        diff = method_data_hard_res[lang] - method_data_prompt_res[lang]
        
        # get index
        # Number of maximum and minimum values to retrieve
        n = 5

        # get good example
        # Get the indices of the top n maximum values
        max_indices = np.argsort(diff)[-n:][::-1]  # Sort and take the last n indices, then reverse
        max_values = diff[max_indices]  # Use these indices to get the values
        # print(max_values)
        for idx, index in enumerate(max_indices):
            good = {}
            good[f'COMET Score'] = max_values[idx]
            good[f'English'] = eval_data[index]['text'] # English text
            good[f'{lang} Ground Truth'] = eval_data[index][f'text_{lang}'] # ground truth result
            
            good[f'{lang} Prompting'] = method_data_prompt[index][f'text_{lang}'] # method translation result
            good[f'{lang} Prompting + Word Alignment'] = method_data_hard[index][f'text_{lang}']
            
            examples_method[f'{lang} Good {idx+1}'] = good

        # get bad example
        # Get the indices of the top n minimum values
        min_indices = np.argsort(diff)[:n]  # Sort and take the first n indices
        min_values = diff[min_indices]  # Use these indices to get the values
        
        for idx, index in enumerate(min_indices):
            bad = {}
            bad[f'COMET Score'] = min_values[idx]
            bad[f'English'] = eval_data[index]['text'] # English text
            bad[f'{lang} Ground Truth'] = eval_data[index][f'text_{lang}'] # ground truth result
            
            bad[f'{lang} Prompting'] = method_data_prompt[index][f'text_{lang}'] # method translation result
            bad[f'{lang} Prompting + Word Alignment'] = method_data_hard[index][f'text_{lang}']
            
            examples_method[f'{lang} Bad {idx+1}'] = bad
        
    examples[model_name] = examples_method

In [3]:
examples

{'nllb': {'Arabic Good 1': {'COMET Score': 0.3516668677330017,
   'English': '  --vocab_file=$BERT_BASE_DIR/vocab.txt \\ \n   --bert_config_file=$BERT_BASE_DIR/bert_config.json \\ \n   --init_checkpoint=$BERT_BASE_DIR/bert_model.ckpt \\ \n   --max_seq_length=128 \\ \n   --train_batch_size=32 \\ \n   --learning_rate=5e-5 \\ \n   --num_train_epochs=2.0 \\ \n   --output_dir=/tmp/xnli_output/ \n ``` \n\n The multilingual model does not require any special consideration or API changes. However, the `BasicTokenizer` in `tokenization.py` should be updated to support Chinese character tokenization.',
   'Arabic Ground Truth': "--vocab_file=$BERT_BASE_DIR/vocab.txt \\ \n   --bert_config_file=$BERT_BASE_DIR/bert_config.json \\ \n   --init_checkpoint=$BERT_BASE_DIR/bert_model.ckpt \\ \n   --max_seq_length=128 \\ \n   --train_batch_size=32 \\ \n   --learning_rate=5e-5 \\ \n   --num_train_epochs=2.0 \\ \n   --output_dir=/tmp/xnli_output/ \n ``` \n\n لا يتطلب النموذج متعدد اللغات أي اعتبارات خاصة أو